In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. **Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context**
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---

# Python Code Tutor with Dynamic Instructions
* "Compute" instructions at runtime
    * Inject live data/runtime state 
        * user role, skill level, permissions, locale, workflow state, ...
    * Avoid near-duplicate agents that differ only in their instructions
* Pass a *callable* as `Agent`'s `instructions=` argument
    * SDK calls it on every run to generate the system prompt fresh
    * **Callable signature**  
      `def func (run_context: RunContextWrapper[T], agent: Agent[T]) -> str`
    * **`RunContextWrapper`**
        * Type `run`'s `context` parameter  
* **Tutor Demo** 
    * Same agent, same code, same question
    * Difference is `level` field in context object
    * Flips system prompt between novice- and developer-level explanations 

---

## Context Type and Dynamic Instructions Function

* `LearnerContext` holds the learner's experience level and the code to explain
* Any Python object can be the context 
    * `dataclass` works well here because it gives type-safe access to `run_context.context` inside the instructions function
* The callable can contain any logic — `if`/`else`, database lookups, f-strings —
  as long as it returns a `str`

---

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

from agents import Agent, RunContextWrapper, Runner, trace
from IPython.display import Markdown, display

@dataclass
class LearnerContext:
    level: Literal['novice', 'experienced']

def tutor_instructions(
    run_context: RunContextWrapper[LearnerContext],
    agent: Agent[LearnerContext],
) -> str:
    """Return a system prompt tailored to the learner's experience level."""
    level = run_context.context.level

    if level == 'novice':
        return (
            'You are a patient Python tutor for beginners. '
            'Explain each concept using simple everyday analogies. '
            'Define every term you use. Avoid jargon and assumed prior '
            'programming knowledge. Use short sentences and concrete examples.'
        )
    else:  # experienced
        return (
            'You are a concise Python tutor for experienced programmers. '
            'Focus on Python-specific features and idioms. '
            'Note what makes the code Pythonic and highlight differences '
            'from other languages where relevant. Skip basic explanations.'
        )

# The agent is created once. The instructions callable is evaluated fresh on
# every Runner.run() call, so a single agent handles both experience levels.
tutor_agent = Agent(
    name='Python Code Tutor',
    instructions=tutor_instructions  # callable, not a string
)

---
## Load Code & Create Prompt
* `dice_game.py` is a compact example that demos
    * `Enum`
    * Tuple packing/unpacking
    * `match`...`case`
    * `while`
    * f-strings
* The same code goes to both agent runs; only context `level` will change

---

In [ ]:
CODE_FILE = Path('resources') / 'dice_game.py'
code = CODE_FILE.read_text()

prompt = f"""Explain this Python code to me, concept by concept:

```python
{code}
```"""

---

## Create the Context and Launch the Agent for Novice Explanation
* Study explanation of `roll_dice` function

In [ ]:
context = LearnerContext(level='novice')

with trace('02-05-10-dynamic-instructions-novice'):
    result = await Runner.run(tutor_agent, prompt, context=context)

display(Markdown(f'## Explanation for a **novice** programmer\n\n{result.final_output}'))

---
## Same Agent, Same Code — Experienced Developer Explanation
* Changing `level` to `'experienced'` is the only difference
* SDK calls `tutor_instructions` again and gets a completely different
  system prompt, producing a more technical explanation
* Study explanation of `roll_dice` function
---

In [ ]:
context = LearnerContext(level='experienced')

with trace('02-05-10-dynamic-instructions-experienced'):
    result = await Runner.run(tutor_agent, prompt, context=context)

display(Markdown(f'## Explanation for an **experienced** programmer\n\n{result.final_output}'))

---
© 2026 by Deitel & Associates, Inc. All Rights Reserved.